# Laboratorio 6 - Análisis de redes sociales (YouTube)

## 1. Carga, comprensión e integración de los datos

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sys.path.append('../src')
from carga_e_integracion import cargar_datos, verificar_llaves_y_unidades, integrar_datos
from limpieza_y_preprocesamiento import diagnostico_calidad, preprocesar_comentarios
from eda import calcular_resumen_descriptivo, analizar_concentracion, generar_graficos_eda
from red_bipartita import construir_red_bipartita, exportar_tablas_nodos_aristas, visualizar_red_bipartita

path_v = '../data/raw/youtube_videos.csv'
path_c = '../data/raw/youtube_comments.csv'

df_v, df_c = cargar_datos(path_v, path_c)
print(f'Videos cargados: {df_v.shape[0]} filas, {df_v.shape[1]} columnas')
print(f'Comentarios cargados: {df_c.shape[0]} filas, {df_c.shape[1]} columnas')

Videos cargados: 293 filas, 20 columnas
Comentarios cargados: 406 filas, 17 columnas


In [2]:
verif = verificar_llaves_y_unidades(df_v, df_c)
print('Resultados de integración:')
for k, v in verif.items():
    print(f' - {k}: {v}')

df_merged = integrar_datos(df_v, df_c)
df_merged.to_csv('../data/processed/dataset_integrado.csv', index=False)
print(f'Dataset integrado guardado con forma: {df_merged.shape}')

Resultados de integración:
 - total_videos: 293
 - pk_video_unica: True
 - total_comentarios: 406
 - pk_comentario_unica: True
 - comentarios_asociados: 406
 - porcentaje_asociados: 100.0
Dataset integrado guardado con forma: (406, 36)


## 2. Calidad, limpieza y preprocesamiento

In [3]:
diag = diagnostico_calidad(df_v, df_c)
print('Diagnóstico de nulos en Comentarios:')
print(diag['comentarios_nulos'])
print('\nVariables constantes en comentarios:', diag['variables_constantes_comentarios'])
print('Duplicados en comentarios:', diag['comentarios_duplicados'])
print('Duplicados en videos:', diag['videos_duplicados'])

Diagnóstico de nulos en Comentarios:
{'video_id': 0, 'comment_id': 0, 'video_title': 0, 'channel_name': 0, 'channel_id': 0, 'author_name': 0, 'author_channel_id': 0, 'text': 0, 'source_query': 0, 'source_group': 0, 'dataset_sources': 0, 'author_handle': 0, 'published_text': 0, 'like_count_text': 0, 'reply_count': 0, 'is_pinned': 0, 'viewer_rating': 406}

Variables constantes en comentarios: ['is_pinned', 'viewer_rating']
Duplicados en comentarios: 0
Duplicados en videos: 0


In [4]:
df_c_proc = preprocesar_comentarios(df_c)
df_c_proc.to_csv('../data/processed/dataset_limpio_comentarios.csv', index=False)

print('Preprocesamiento completado. Resumen de likes limpios:')
print(df_c_proc['like_count'].describe())

vacios_limpio = (df_c_proc['texto_limpio'] == '').sum()
print(f'Comentarios que quedaron vacíos tras la limpieza de stopwords/símbolos: {vacios_limpio}')

Preprocesamiento completado. Resumen de likes limpios:
count    406.000000
mean       5.726601
std       30.661298
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max      405.000000
Name: like_count, dtype: float64
Comentarios que quedaron vacíos tras la limpieza de stopwords/símbolos: 7


## 3. Análisis exploratorio (EDA)

In [5]:
resumen = calcular_resumen_descriptivo(df_v, df_c_proc)
for k, v in resumen.items():
    print(f'{k}: {v}')

conc = analizar_concentracion(df_c_proc)
print('\nConcentración de participación:')
for k, v in conc.items():
    print(f'{k}: {v:.2f}%')

total_videos: 293
total_canales: 97
total_comentarios: 406
total_autores: 332
promedio_videos_por_canal: 3.020618556701031
max_videos_por_canal: 32
promedio_comentarios_por_video: 21.36842105263158
max_comentarios_por_video: 161
total_visualizaciones: 17706015
mediana_visualizaciones: 1175.0

Concentración de participación:
top1_video_pct: 39.66%
top3_videos_pct: 63.05%
top5_videos_pct: 75.37%
top10_autores_pct: 10.34%


In [6]:
generar_graficos_eda(df_v, df_c_proc, '../reports/figures')
print('Gráficos guardados en reports/figures/')

Gráficos guardados en reports/figures/


## 4. Construcción de la red bipartita autor-video

In [7]:
B, edge_weights = construir_red_bipartita(df_c_proc, df_v)
print(f'Red Bipartita Construida:')
print(f' - Total de nodos: {B.number_of_nodes()}')
print(f' - Nodos de Autores: {len([n for n, d in B.nodes(data=True) if d["node_type"] == "author"])}')
print(f' - Nodos de Videos: {len([n for n, d in B.nodes(data=True) if d["node_type"] == "video"])}')
print(f' - Total de Aristas (co-participaciones): {B.number_of_edges()}')

Red Bipartita Construida:
 - Total de nodos: 351
 - Nodos de Autores: 332
 - Nodos de Videos: 19
 - Total de Aristas (co-participaciones): 343


In [8]:
nodes_df, edges_df = exportar_tablas_nodos_aristas(B, df_c_proc, df_v, '../data/processed')
print('Tabla de nodos (primeras 5 filas):')
print(nodes_df.head())
print('\nTabla de aristas (primeras 5 filas):')
print(edges_df.head())

Tabla de nodos (primeras 5 filas):
                    node_id node_type  ... degree total_comments
0  UCdFlugHJJa4l3YqWuNRmvXw    author  ...      2              2
1  UCvl1tzQeBeGy6efPTRJXSCw    author  ...      1              1
2  UCRAquv8el-tQ30bN7MlmySQ    author  ...      1              1
3  UCkbsS_3D-pvg1iHOld9uvIg    author  ...      1              1
4  UCOLHH4ZpMxPYF-Q6e6wvn5A    author  ...      1              1

[5 rows x 6 columns]

Tabla de aristas (primeras 5 filas):
                     source       target  weight
0  UC-HeUTT6_g-VoiWds2a4H-w  6W4u8sGEnGM       1
1  UC-Iul5tDYH_oiAMXQH-KaaQ  6W4u8sGEnGM       1
2  UC-QOpE7GOxlXcHZ8b7HbQKA  OkXlHx0hx-8       2
3  UC-fiZBS5Gp1eCJd9ucgWnCg  j43HgwYFKfk       1
4  UC-hfX8J5JPJ-dfuakzegMLA  n8iP75gIpmw       1


In [9]:
visualizar_red_bipartita(B, df_v, '../reports/figures/red_bipartita_autor_video.png')
print('Visualización guardada en reports/figures/red_bipartita_autor_video.png')

Visualización guardada en reports/figures/red_bipartita_autor_video.png
